# Noonchi Translator — mBART Fine-tuning

Fine-tunes `facebook/mbart-large-50-many-to-many-mmt` on the formality-labeled EN–KR corpus.

**Before running:** Set Runtime → Change runtime type → GPU (T4 or better).

**Training time estimates:**
- 50K rows, 3 epochs, batch 4 × grad_accum 8 on T4: ~5–8 hours
- Full 423K rows: use A100 (Colab Pro) or a university cluster

All checkpoints are saved to Google Drive so a session disconnect doesn't lose progress.

In [1]:
# Cell 1 — Verify GPU is assigned before doing anything else
!nvidia-smi

Thu Apr 30 02:16:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Mount Google Drive
# All model checkpoints write here so disconnects don't lose progress.
from google.colab import drive
drive.mount('/content/drive')

import os

Mounted at /content/drive


In [3]:
# Cell 3 — Install dependencies
# transformers 4.41+ required for eval_strategy rename and text_target tokenizer API.
# accelerate is a hard requirement of HuggingFace Trainer (often missing from requirements).
# mecab-ko (via konlpy install script) is needed for formality_accuracy evaluation.
!pip install -q 'transformers>=4.41.0' 'accelerate>=0.26.0' datasets sentencepiece sacrebleu
!pip install -q mecab-python3 konlpy
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 94.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.0/438.0 kB 41.8 MB/s eta 0:00:00
Install mecab-ko
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1381k  100 1381k    0     0   430k      0  0:00:03  0:00:03 --:--:--  885k
mecab-0.996-ko-0.9.2/
mecab-0.996-ko-0.9.2/example/
mecab-0.996-ko-0.9.2/example/example.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.c
mecab-0.996-ko-0.9.2/example/example.c
mecab-0.996-ko-0.9.2/example/thread_test.cpp
mecab-0.996-ko-0.9.2/mecab-config.in
mecab-0.996-ko-0.9.2/man

In [ ]:
# Cell 4 — Clone repo and set up Python path
# Add your GitHub token to Colab Secrets (key icon in sidebar) as GITHUB_TOKEN
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

import os, subprocess, sys, importlib

repo_path = '/content/noonchi-translator'
if os.path.exists(repo_path):
    subprocess.run(['git', '-C', repo_path, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone',
        f'https://{token}@github.com/cyatco01/noonchi-translator.git', repo_path],
        check=True)

sys.path.insert(0, repo_path)
importlib.invalidate_caches()
%cd /content/noonchi-translator

from backend.model.train import load_model_and_tokenizer
from backend.model.dataset import load_split
print('Imports OK')


In [8]:
# Cell 5 — Copy training data from Drive
# Upload train.tsv and val.tsv to /content/drive/MyDrive/noonchi/ first.
# These files are too large to commit to git (~50MB each).
!cp /content/drive/MyDrive/noonchi/train.tsv data/train.tsv
!cp /content/drive/MyDrive/noonchi/val.tsv data/val.tsv
!cp /content/drive/MyDrive/noonchi/test.tsv data/test.tsv

# Verify line counts
!wc -l data/train.tsv data/val.tsv data/test.tsv

  338959 data/train.tsv
   42370 data/val.tsv
   42373 data/test.tsv
  423702 total


In [9]:
# Cell 6 — Smoke test: load model and one batch before committing to full training
# If this fails, debug here rather than 30 minutes into a training run.
model, tokenizer = load_model_and_tokenizer()
ds = load_split('data/val.tsv', tokenizer)
sample = ds[0]
print('Dataset size:', len(ds))
print('Sample keys:', list(sample.keys()))
print('input_ids length:', len(sample['input_ids']))
print('labels length:', len(sample['labels']))
print('decoder_start_token_id:', model.config.decoder_start_token_id)
print('forced_bos_token_id:', model.config.forced_bos_token_id)
del model  # free VRAM before training

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Dataset size: 42369
Sample keys: ['input_ids', 'attention_mask', 'labels']
input_ids length: 13
labels length: 14
decoder_start_token_id: 250014
forced_bos_token_id: None


In [ ]:
# if run time disconnected or edits made to git
!git -C /content/noonchi-translator pull     

In [10]:
# Cell 7 — Train
# --max-rows 50000: stratified 50K sample fits a free T4 session (~5–8 hours, 3 epochs)
# Remove --max-rows to train on the full 423K dataset (needs A100 or ~40+ hours)
# Add --resume to pick up from the latest checkpoint if the session crashed
# Cell 7 — Train
# --max-rows 50000: stratified 50K sample fits a free T4 session (~5–8 hours, 3 epochs)
# Remove --max-rows to train on the full 423K dataset (needs A100 or ~40+ hours)
# Add --resume to pick up from the latest checkpoint if the session crashed
!python -m backend.model.train --data data/train.tsv --output /content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart --max-rows 50000

INFO: HTTP Request: HEAD https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggi

In [ ]:
# #if runtime disconnects
# !cd /content/noonchi-translator && git pull                                     
# import sys                                                                      
# sys.path.insert(0, '/content/noonchi-translator')                               
# %cd /content/noonchi-translator    

In [ ]:
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)   

In [ ]:
# Cell 8 — Evaluate on held-out test split
# Sanity bars (50K model): chrF > 20, formality_accuracy > 0.60
# Target bars (full model): chrF > 30, formality_accuracy > 0.80
!python -m backend.model.evaluate \\
    --model /content/drive/MyDrive/noonchi/checkpoints/noonchi-mbart \\
    --test data/test.tsv \\
    --batch-size 8 \\
    --num-beams 4 \\
    --hypotheses-cache /content/drive/MyDrive/noonchi/hypotheses_cache.txt \\
    --output /content/drive/MyDrive/noonchi/eval_results.json


In [ ]:
# Cell 9 — Inspect sample predictions
# Run this after Cell 8 to sanity-check what the model actually generated.
# If outputs are empty, garbage, or all the same language as input, something is wrong upstream.
cache_path = "/content/drive/MyDrive/noonchi/hypotheses_cache.txt"
test_path = "data/test.tsv"

import csv
hypotheses = open(cache_path, encoding="utf-8").read().splitlines()

references, en_inputs = [], []
with open(test_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    next(reader)
    for row in reader:
        if len(row) == 3:
            en_inputs.append(row[0])
            references.append(row[1])

print(f"Total hypotheses: {len(hypotheses)}, references: {len(references)}\n")
for i in range(min(10, len(hypotheses))):
    print(f"[{i}] EN:  {en_inputs[i]}")
    print(f"     REF: {references[i]}")
    print(f"     HYP: {hypotheses[i]}")
    print()
